In [1]:
# ============================================================
# Install Dependencies
# ============================================================

!pip install -q --upgrade pip

!pip install -q \
unsloth \
transformers \
trl \
datasets \
accelerate \
bitsandbytes \
peft \
sentencepiece

print("=" * 70)
print("Dependencies Installed Successfully")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.2 MB/s eta 0:00:00
Dependencies Installed Successfully


In [2]:
# ============================================================
# Imports
# ============================================================

import os
import time
import random
import numpy as np
import torch

from datasets import load_dataset

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

from trl import SFTTrainer, SFTConfig

# ============================================================
# Configuration
# ============================================================

# Model
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"

# Or use a local model:
# BASE_MODEL = "/kaggle/input/llms/Meta-Llama-3.1-8B-Instruct"

MAX_SEQ_LENGTH = 7608
LOAD_IN_4BIT = True
DTYPE = None

# Dataset
DATASET_PATH = "/kaggle/input/datasets/adityammantri/cybersecurity-red-team-and-blue-team-sft-dataset/train_blue.jsonl"

# LoRA
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# Training
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 3

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

LOGGING_STEPS = 10

OUTPUT_DIR = "./outputs"
SAVE_TOTAL_LIMIT = 5
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42

# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("Configuration Loaded")
print("=" * 70)

print(f"Model           : {BASE_MODEL}")
print(f"Sequence Length : {MAX_SEQ_LENGTH}")
print(f"Dataset         : {DATASET_PATH}")
print(f"Epochs          : {EPOCHS}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Gradient Accum  : {GRAD_ACCUM}")
print("=" * 70)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Configuration Loaded
Model           : unsloth/Qwen2.5-7B-Instruct
Sequence Length : 7608
Dataset         : /kaggle/input/datasets/adityammantri/cybersecurity-red-team-and-blue-team-sft-dataset/train_blue.jsonl
Epochs          : 3
Batch Size      : 1
Gradient Accum  : 16


In [3]:
# ============================================================
# Load Base Model
# ============================================================

print("=" * 70)
print("Loading Base Model...")
print("=" * 70)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("=" * 70)
print("Base Model Loaded Successfully")
print("=" * 70)

print(f"Model Name      : {BASE_MODEL}")
print(f"Sequence Length : {MAX_SEQ_LENGTH}")
print(f"4-bit Loading   : {LOAD_IN_4BIT}")

print("=" * 70)

Loading Base Model...
==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base Model Loaded Successfully
Model Name      : unsloth/Qwen2.5-7B-Instruct
Sequence Length : 7608
4-bit Loading   : True


In [4]:
# ============================================================
# Configure LoRA
# ============================================================

print("=" * 70)
print("Applying LoRA Adapters...")
print("=" * 70)

model = FastLanguageModel.get_peft_model(

    model,

    r=LORA_RANK,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

    use_rslora=False,

    loftq_config=None,

    target_modules=TARGET_MODULES,

)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("=" * 70)
print("LoRA Successfully Applied")
print("=" * 70)

print(f"Trainable Parameters : {trainable_params:,}")
print(f"Total Parameters     : {total_params:,}")
print(f"LoRA Rank            : {LORA_RANK}")
print(f"LoRA Alpha           : {LORA_ALPHA}")
print(f"LoRA Dropout         : {LORA_DROPOUT}")

print("=" * 70)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Applying LoRA Adapters...


Unsloth 2026.7.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA Successfully Applied
Trainable Parameters : 80,740,352
Total Parameters     : 4,972,287,488
LoRA Rank            : 32
LoRA Alpha           : 64
LoRA Dropout         : 0.05


In [5]:
# ============================================================
# Load Dataset
# ============================================================

print("=" * 70)
print("Loading Dataset...")
print("=" * 70)

dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
    split="train",
)

print("=" * 70)
print("Dataset Loaded Successfully")
print("=" * 70)

print(f"Total Samples : {len(dataset)}")

print("\nDataset Features:")
print(dataset.features)

print("=" * 70)

Loading Dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Dataset Loaded Successfully
Total Samples : 551

Dataset Features:
{'scenario_id': Value('string'), 'example_id': Value('string'), 'messages': List({'role': Value('string'), 'content': Value('string')})}


In [6]:
# ============================================================
# Prepare Dataset
# ============================================================

dataset = dataset.shuffle(seed=SEED)

print("=" * 70)
print("Dataset Prepared")
print("=" * 70)

print(f"Training Samples : {len(dataset)}")

print("\nFirst Conversation:\n")

for message in dataset[0]["messages"]:
    print(f"{message['role'].upper()}:")
    print(message["content"])
    print()

print("=" * 70)

Dataset Prepared
Training Samples : 551

First Conversation:

SYSTEM:
You are a Blue Team cybersecurity analyst specializing in architectural security assessments of enterprise software systems.

Your objective is to identify the single most effective defensive mitigation that addresses the root cause of the observed attack using the provided business context, architecture, workflow, trust boundaries, and source code.

Perform a structured architectural analysis before reaching a conclusion.

During your analysis:

• Identify the canonical security boundary of the application.
• Follow the end-to-end business workflow.
• Track how security-critical information is transformed between components.
• Distinguish architectural weaknesses from implementation details.
• Eliminate competing defensive hypotheses before selecting the final defense.
• Base every conclusion only on the supplied information.

Do not assume undocumented behavior.

Do not invent vulnerabilities, mitigations, or archi

In [7]:
def formatting_prompts_func(examples):

    conversations = examples["messages"]

    texts = [

        tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )

        for conversation in conversations

    ]

    return {

        "text": texts,

    }
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatting conversations",
)

print("=" * 70)
print("Chat Template Applied Successfully")
print("=" * 70)

Formatting conversations:   0%|          | 0/551 [00:00<?, ? examples/s]

Chat Template Applied Successfully


In [8]:
# ============================================================
# Create Trainer
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    dataset_text_field="text",

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    num_train_epochs=EPOCHS,

    logging_steps=LOGGING_STEPS,

    save_strategy="epoch",

    save_total_limit=SAVE_TOTAL_LIMIT,

    optim="adamw_8bit",

    lr_scheduler_type="cosine",

    bf16=torch.cuda.is_bf16_supported(),

    fp16=not torch.cuda.is_bf16_supported(),

    seed=SEED,

    report_to="none",
)

trainer = SFTTrainer(

    model=model,

    tokenizer=tokenizer,

    train_dataset=dataset,

    args=training_args,

)

# Train only on assistant responses
trainer = train_on_responses_only(

    trainer,

    instruction_part="<|im_start|>user\n",

    response_part="<|im_start|>assistant\n",

)

print("=" * 70)
print("Trainer Ready")
print("=" * 70)

print(f"Training Samples : {len(dataset):,}")
print(f"Epochs           : {EPOCHS}")
print(f"Learning Rate    : {LEARNING_RATE}")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Gradient Accum   : {GRAD_ACCUM}")

print("=" * 70)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/551 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Filter:   0%|          | 0/551 [00:00<?, ? examples/s]

Unsloth: Removed 177 out of 551 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.
Trainer Ready
Training Samples : 551
Epochs           : 3
Learning Rate    : 0.0002
Batch Size       : 1
Gradient Accum   : 16


In [9]:
# ============================================================
# Train Model
# ============================================================

print("=" * 70)
print("Starting Supervised Fine-Tuning...")
print("=" * 70)

start_time = time.time()

trainer_stats = trainer.train()

end_time = time.time()

training_time = end_time - start_time

print("=" * 70)
print("Training Completed Successfully")
print("=" * 70)

print(f"Training Time : {training_time/60:.2f} minutes")
print(f"Training Time : {training_time/3600:.2f} hours")

if hasattr(trainer_stats, "training_loss"):
    print(f"Final Training Loss : {trainer_stats.training_loss:.4f}")

print("=" * 70)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Supervised Fine-Tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 374 | Num Epochs = 3 | Total steps = 72
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.039823
20,1.516371
30,1.196252
40,1.057746
50,1.041851
60,0.783684
70,0.769987


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-48/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-72/tokenizer_config.json.


Training Completed Successfully
Training Time : 288.89 minutes
Training Time : 4.81 hours
Final Training Loss : 1.1872


In [10]:
# ============================================================
# Save Models
# ============================================================

LORA_OUTPUT = os.path.join(OUTPUT_DIR, "lora_adapter")
MERGED_OUTPUT = os.path.join(OUTPUT_DIR, "merged_model")

print("=" * 70)
print("Saving LoRA Adapter...")
print("=" * 70)

model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)

print("LoRA adapter saved.")

print("=" * 70)
print("Saving Merged 16-bit Model...")
print("=" * 70)

model.save_pretrained_merged(
    MERGED_OUTPUT,
    tokenizer,
    save_method="merged_16bit",
)

print("=" * 70)
print("Model Saved Successfully")
print("=" * 70)

print(f"LoRA Adapter : {LORA_OUTPUT}")
print(f"Merged Model : {MERGED_OUTPUT}")

print("=" * 70)

Saving LoRA Adapter...


Unsloth: Restored added_tokens_decoder metadata in ./outputs/lora_adapter/tokenizer_config.json.


LoRA adapter saved.
Saving Merged 16-bit Model...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:49<00:00, 12.38s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:55<00:00, 28.86s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/outputs/merged_model`
Model Saved Successfully
LoRA Adapter : ./outputs/lora_adapter
Merged Model : ./outputs/merged_model
